In [1]:
45


45

In [2]:
print("hello")

hello


In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader


In [4]:
load_dotenv()

True

In [5]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")

In [6]:
DATA_FILE_PATH=os.path.join("data", "hr_policy.txt")

DATA INGESTION


In [8]:
loader = TextLoader(DATA_FILE_PATH, encoding='utf-8')
documents = loader.load()
print(documents) 

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

LANGCHAIN DOCUMENT

LANCHAIN PROCESS EVERYTHING IN DOCUMENT 

DOCUMENT :

PAGE CONTENT

META DATA

In [9]:
len(documents)

1

In [13]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


SPLITTING DOCUMENTS

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [18]:
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [19]:
print(len(chunks))

9


NOW EACH SPLITTED CHUNK IS DOCUMENT

In [20]:
print(chunks[4])

page_content='4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final settlement
processed within 45 days of the last working day.' metadata={'source': 'data\\hr_policy.txt'}


In [22]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


EMBEDDING

In [23]:
from langchain_community.embeddings import JinaEmbeddings

In [24]:
embeddings_model=JinaEmbeddings(
    model_name='jina-embeddings-v5-text-small'
)

STORE DATA IN VECTOR DB

In [25]:
from langchain_community.vectorstores import FAISS

In [26]:
vector_store=FAISS.from_documents(chunks, embeddings_model)


In [27]:
test_query="How many sick leave do employees get"
top_matches= vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,1):
    print(f"----Match{i}------")
    print(match.page_content)

Query: How many sick leave do employees get

----Match1------
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
----Match2------
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


LLM

In [28]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")

)

In [30]:
test_response=llm.invoke("explain ML")
test_response.content

'**Machine Learning (ML) – A Quick Overview**\n\n---\n\n### 1. What is Machine Learning?\nMachine Learning is a branch of artificial intelligence (AI) that gives computers the ability to **learn from data** and improve their performance on a task without being explicitly programmed for every possible scenario.\n\n- **Traditional programming:** You write rules → computer follows them.\n- **ML approach:** You give the computer data → it discovers patterns → it creates a model that can make predictions or decisions on new, unseen data.\n\n---\n\n### 2. Core Concepts\n\n| Term | Meaning |\n|------|----------|\n| **Data** | The raw information (numbers, text, images, etc.) that the algorithm learns from. |\n| **Feature** | A measurable property or attribute of the data (e.g., age, pixel intensity, word count). |\n| **Label** | The “answer” you want the model to predict (e.g., spam/not‑spam, house price). |\n| **Model** | The mathematical representation (set of parameters) that maps features

AI AGENT
LLM BRAIN 
TOOL DATA

In [31]:
from langchain.agents import create_agent

In [37]:
from langchain.tools import tool

@tool
def search_hr_policy(query: str) -> str:
    """
    Search the HR policy knowledge base for information
    about employee policies, leaves, salary, attendance,
    benefits and other HR-related questions.
    """

    results = vector_store.similarity_search(
        query,
        k=3
    )

    if not results:
        return "No relevant HR policy information found."

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    return context

In [38]:
result = search_hr_policy.invoke({
    "query": "How many sick leaves do employees get?"
})

print(result)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Performance is reviewed at the end of the probation period to confirm employment.


In [39]:
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=""" 
                    You are a friendly HR assistant.

Always use the search_hr_policy tool to look up facts before answering.

If the answer isn't in the search results, say you don't know instead of guessing.
"""

)

In [40]:
response = hr_assistant.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How many sick leaves do employees get?"
        }
    ]
})

print(response["messages"][-1].content)

Employees are entitled to **10 paid sick days per year**. (Source: company leave policy – “Sick leave is separate from annual leave, and employees get 10 paid sick days per year.”)
